# Bài tập 1: Dự báo Tiêu thụ Năng lượng PJME sử dụng ARIMA

Dataset: `PJME_hourly.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_pacf, plot_acf
from statsmodels.tsa.arima.model import ARIMA
from pmdarima.arima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import warnings
warnings.filterwarnings("ignore")

def adf_test(x):
    indices = ['ADF: Test statistic', 'p value', '# of Lags', '# of Observations']
    test = adfuller(x, autolag='AIC')
    results = pd.Series(test[:4], index=indices)
    for key, value in test[4].items():
        results[f'Critical Value ({key})'] = value
    return results

def kpss_test(x):
    indices = ['KPSS: Test statistic', 'p value', '# of Lags']
    test = kpss(x)
    results = pd.Series(test[:3], index=indices)
    for key, value in test[3].items():
        results[f'Critical Value ({key})'] = value
    return results

## 1. Load và Khám phá Dữ liệu

In [ ]:
df = pd.read_csv('data/PJME_hourly.csv', index_col='Datetime', parse_dates=True)
df = df.resample('D').mean() # Chuyển sang dữ liệu ngày để giảm độ nhiễu
df_log = np.log(df['PJME_MW'])
df_log.plot(figsize=(12,6), title='Log PJME Energy Consumption')

## a) Phân rã dữ liệu (Decomposition) và Nhận xét

In [ ]:
decomposition = seasonal_decompose(df_log, model='additive', period=365)
fig = decomposition.plot()
fig.set_size_inches(12, 8)
plt.show()

**Nhận xét:**
- **Trend:** Tiêu thụ năng lượng có xu hướng ổn định trong dài hạn nhưng có các dao động nhẹ qua các năm.
- **Seasonal:** Tính mùa vụ hàng năm rất mạnh, phản ánh nhu cầu sử dụng điện khác nhau giữa các mùa trong năm.
- **Resid:** Các thành phần không quy luật phản ánh các biến động ngắn hạn hoặc sự kiện đặc biệt.

## b) Kiểm định tính dừng và Tương quan

In [ ]:
print("ADF Test:", adf_test(df_log))
print("\nKPSS Test:", kpss_test(df_log))

# ACF and PACF
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 4))
plot_acf(df_log, ax=ax1, lags=40)
plot_pacf(df_log, ax=ax2, lags=40)
plt.show()

## c) Fit mô hình ARIMA (80% Train), Đánh giá và Dự báo tương lai

In [ ]:
train_size = int(len(df_log) * 0.8)
train, test = df_log[:train_size], df_log[train_size:]

# Tìm p, d, q
auto_model = auto_arima(train, seasonal=False, trace=True)
order = auto_model.order
print(f"Best Order: {order}")

# Fit model
model = ARIMA(train, order=order)
model_fit = model.fit()

# Dự báo trên tập test
fc = model_fit.forecast(len(test))
rmse = np.sqrt(mean_squared_error(test, fc))
mape = mean_absolute_percentage_error(test, fc)
print(f"Test RMSE: {rmse}, MAPE: {mape}")

# Dự báo tương lai 30 ngày
plt.figure(figsize=(12,6))
plt.plot(test.index, np.exp(test), label='Actual')
plt.plot(test.index, np.exp(fc), label='Forecast')
plt.title('PJME Energy Forecast (Standard 80/20)')
plt.legend()
plt.show()

## d) Expanding Window và Sliding Window

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# Expanding Window
tscv = TimeSeriesSplit(n_splits=5)
exp_errors = []
for train_idx, test_idx in tscv.split(df_log):
    cv_train, cv_test = df_log.iloc[train_idx], df_log.iloc[test_idx]
    m = ARIMA(cv_train, order=order).fit()
    preds = m.forecast(len(cv_test))
    exp_errors.append(np.sqrt(mean_squared_error(cv_test, preds)))

# Sliding Window
window = 3000
slid_errors = []
for i in range(0, len(df_log) - window - 100, 500):
    cv_train = df_log.iloc[i : i+window]
    cv_test = df_log.iloc[i+window : i+window+100]
    m = ARIMA(cv_train, order=order).fit()
    preds = m.forecast(len(cv_test))
    slid_errors.append(np.sqrt(mean_squared_error(cv_test, preds)))

print(f"Expanding RMSE: {np.mean(exp_errors)}")
print(f"Sliding RMSE: {np.mean(slid_errors)}")